In [1]:
%load_ext autoreload
%autoreload 2

import sys
ROOT = r"C:\Users\barbe\Growing-virtual-materials-cellular-automata-meets-geostatistics"
sys.path.insert(0, ROOT)

from pathlib import Path
print("pgsca exists:", (Path(ROOT) / "pgsca" / "__init__.py").exists())

pgsca exists: True


In [2]:
import inspect
from pgsca.hybrid_tools import estimate_anisotropy_masked
print('n_angles' in inspect.signature(estimate_anisotropy_masked).parameters)  # True = new code

True


In [3]:
import numpy as np
from pgsca import (make_gaussian_fields, make_lithotype_map,
                   recover_plurigaussian)
from pgsca.hybrid_tools import estimate_anisotropy

In [4]:
import inspect
print("ellipse version live:",
      'n_angles' in inspect.signature(estimate_anisotropy).parameters)
# True  -> new ellipse code is loaded
# False -> old code; save the file, restart kernel, re-run Cells 0-6

ellipse version live: True


In [5]:
def make_known_map(grid_size, params_1, params_2, proportions, seed_1=0, seed_2=1):
    """3-phase lithotype map from KNOWN plurigaussian params.
    params_i = (alpha_rad, L_major, L_minor). Returns (known_map, truth_dict)."""
    a1, Lmaj1, Lmin1 = params_1
    a2, Lmaj2, Lmin2 = params_2
    f1, f2 = make_gaussian_fields(
        grid_size=grid_size,
        len_scale_1=[Lmaj1, Lmin1], angles_1=[a1, a1], seed_1=seed_1,
        len_scale_2=[Lmaj2, Lmin2], angles_2=[a2, a2], seed_2=seed_2,
    )
    p0, p1, p2 = proportions
    m = make_lithotype_map(f1, f2, Mat1=p0, Mat2=p1, Mat3=p2).astype(int)
    truth = dict(proportions=list(proportions), params_1=params_1, params_2=params_2)
    return m, truth

In [6]:
def angle_err_deg(a_true, a_est):
    """Smallest angle between two orientations, in degrees (mod 180)."""
    d = abs(a_true - a_est) % np.pi
    return np.degrees(min(d, np.pi - d))

def evaluate_recovery(known_map, truth, **rec_kwargs):
    """Single-run: full recovery vs truth, printed side by side."""
    rec = recover_plurigaussian(known_map, **rec_kwargs)
    print("proportions  true:", np.round(truth['proportions'], 3),
          "  recovered:", np.round(rec['proportions'], 3))
    for k in ('params_1', 'params_2'):
        at, Lmaj_t, Lmin_t = truth[k]
        ae, Lmaj_e, Lmin_e = rec[k]
        print(f"\n{k}")
        print(f"  alpha    true {np.degrees(at):6.1f}   recovered {np.degrees(ae):6.1f}"
              f"   err {angle_err_deg(at, ae):5.1f} deg")
        print(f"  L_major  true {Lmaj_t:6.1f}   recovered {Lmaj_e:6.1f}"
              f"   err {Lmaj_e - Lmaj_t:+6.1f}")
        print(f"  L_minor  true {Lmin_t:6.1f}   recovered {Lmin_e:6.1f}"
              f"   err {Lmin_e - Lmin_t:+6.1f}")
    return rec

In [7]:
def recovery_errors(truth_params, n_seeds=6, base_seed=0, **rec_kwargs):
    """Recovery over several realisations of the SAME params -> separates
    systematic bias (mean error) from estimator noise (std)."""
    rows = {'params_1': [], 'params_2': []}
    for s in range(n_seeds):
        known_map, truth = make_known_map(
            seed_1=2*s + base_seed, seed_2=2*s + base_seed + 1, **truth_params)
        rec = recover_plurigaussian(known_map, **rec_kwargs)
        for k in ('params_1', 'params_2'):
            at, Lmaj_t, Lmin_t = truth[k]
            ae, Lmaj_e, Lmin_e = rec[k]
            rows[k].append((angle_err_deg(at, ae), Lmaj_e - Lmaj_t, Lmin_e - Lmin_t))
        print(f"seed set {s+1}/{n_seeds} done")

    for k in ('params_1', 'params_2'):
        arr = np.array(rows[k])          # cols: angle_err, dL_major, dL_minor
        print(f"\n{k}  (n={len(arr)})")
        print(f"  angle err    mean {arr[:,0].mean():5.1f}   std {arr[:,0].std():4.1f}   deg")
        print(f"  L_major err  mean {arr[:,1].mean():+5.1f}   std {arr[:,1].std():4.1f}")
        print(f"  L_minor err  mean {arr[:,2].mean():+5.1f}   std {arr[:,2].std():4.1f}")
    return rows

In [8]:
# reload check should print True, then:
arr2_new = field2_errors(110, 24.0, 10.0, n_seeds=6)   # baseline was angle 8.6 / std 3.3

NameError: name 'field2_errors' is not defined

In [ ]:
def field1_errors(alpha_deg, L_major, L_minor, p0=0.37,
                  grid_size=256, n_seeds=6, **kw):
    """A/B just field 1 (estimate_anisotropy on the phase-0 indicator).
    Skips the slow masked field-2 recovery, which we haven't changed."""
    a_true = np.radians(alpha_deg)
    errs = []
    for s in range(n_seeds):
        f1, f2 = make_gaussian_fields(
            grid_size=grid_size,
            len_scale_1=[L_major, L_minor], angles_1=[a_true, a_true], seed_1=s,
            len_scale_2=[20, 20], angles_2=[0, 0], seed_2=100 + s)
        m  = make_lithotype_map(f1, f2, Mat1=p0, Mat2=(1-p0)/2, Mat3=(1-p0)/2).astype(int)
        B0 = (m == 0).astype(int)                      # phase-0 = field 1 alone
        ae, Lmaj_e, Lmin_e = estimate_anisotropy(B0, **kw)[:3]
        errs.append((angle_err_deg(a_true, ae), Lmaj_e - L_major, Lmin_e - L_minor))
        print(f"{s+1}/{n_seeds}")
    arr = np.array(errs)
    print(f"\nangle  mean {arr[:,0].mean():5.1f}  std {arr[:,0].std():4.1f}  deg")
    print(f"Lmaj   mean {arr[:,1].mean():+5.1f}  std {arr[:,1].std():4.1f}")
    print(f"Lmin   mean {arr[:,2].mean():+5.1f}  std {arr[:,2].std():4.1f}")
    return arr

In [9]:
def field2_errors(alpha2_deg, L_major, L_minor, proportions=(0.37, 0.25, 0.38),
                  grid_size=256, n_seeds=6, **kw):
    """A/B just field 2 (estimate_anisotropy_masked). Field 1 fixed at 30 deg, L 25/12."""
    a2 = np.radians(alpha2_deg)
    p0, p1, p2 = proportions
    errs = []
    for s in range(n_seeds):
        f1, f2 = make_gaussian_fields(
            grid_size=grid_size,
            len_scale_1=[25, 12], angles_1=[np.radians(30), np.radians(30)], seed_1=s,
            len_scale_2=[L_major, L_minor], angles_2=[a2, a2], seed_2=100 + s)
        m    = make_lithotype_map(f1, f2, Mat1=p0, Mat2=p1, Mat3=p2).astype(int)
        ae, Lmaj_e, Lmin_e = estimate_anisotropy_masked((m == 2).astype(int), m != 0, **kw)
        errs.append((angle_err_deg(a2, ae), Lmaj_e - L_major, Lmin_e - L_minor))
        print(f"{s+1}/{n_seeds}")
    arr = np.array(errs)
    print(f"\nangle  mean {arr[:,0].mean():5.1f}  std {arr[:,0].std():4.1f}  deg")
    print(f"Lmaj   mean {arr[:,1].mean():+5.1f}  std {arr[:,1].std():4.1f}")
    print(f"Lmin   mean {arr[:,2].mean():+5.1f}  std {arr[:,2].std():4.1f}")
    return arr

arr2_new = field2_errors(110, 24.0, 10.0, n_seeds=6)

1/6
2/6
3/6
4/6
5/6
6/6

angle  mean   4.3  std  2.2  deg
Lmaj   mean  -1.2  std  2.9
Lmin   mean  -0.3  std  1.4


In [10]:
truth_params = dict(
    grid_size=256,
    params_1=(np.radians(30),  25.0, 12.0),
    params_2=(np.radians(110), 18.0, 15.0),
    proportions=(0.37, 0.25, 0.38),
)
truth_params_aniso = dict(
    grid_size=256,
    params_1=(np.radians(30),  25.0, 12.0),   # ratio ~2.1
    params_2=(np.radians(110), 24.0, 10.0),   # ratio ~2.4 (clearly anisotropic)
    proportions=(0.37, 0.25, 0.38),
)

In [11]:
# field 1 = 30 deg, L 25/12 -- baseline from Cell 5 was angle mean 6.7, std 3.1
arr_new = field1_errors(30, 25.0, 12.0, n_seeds=6)

NameError: name 'field1_errors' is not defined